# Labwork 1 — Losses, likelihood, and gradient checking

**Week 2 · Day 1 · ≈ 170 min at the keyboard**

Read Lecture 1 first. This labwork fills in part of the `optlab` package you cloned;
you edit the real source files, and the notebook checks your work as you go.

**What you build today:** the verification tool you will use all week, then the loss classes every later day minimizes

**Files you will open:**

- `numerics/gradcheck.py`
- `losses.py`
- `problems/glm.py`, `problems/quadratic.py`, `problems/rosenbrock.py`
- `datasets/load.py`

> **The rule.** `src/optlab/interfaces.py`, `results.py`, `errors.py` and `types.py` are
> **provided** — never edit them. Everything else under `src/optlab/` is yours: replace
> each `raise NotImplementedError` with working code, keeping the signature and honouring
> the docstring.

## Setup

Run this once. It points the notebook at your `optlab` clone and gives you a
`check()` helper that runs a specific test file and reports what happened.

In [ ]:
import subprocess
import sys
from pathlib import Path

# Adjust if your clone lives elsewhere.
OPTLAB = (Path.cwd() / ".." / ".." / "optlab").resolve()
assert OPTLAB.exists(), f"optlab not found at {OPTLAB} -- edit OPTLAB above"
print("optlab:", OPTLAB)


def check(*pytest_args):
    """Run pytest inside the optlab clone and show a short report."""
    r = subprocess.run([sys.executable, "-m", "pytest", "-q", "--no-header", *pytest_args],
                       cwd=OPTLAB, capture_output=True, text=True)
    out = r.stdout + r.stderr
    tail = [l for l in out.splitlines() if l.strip()][-12:]
    print("\n".join(tail))
    if "No module named pytest" in out:
        verdict = "pytest is not installed -- run:  pip install -e '.[dev]'"
    elif r.returncode == 0:
        verdict = "PASSED"
    elif r.returncode == 5:
        # Exit code 5 means pytest collected nothing at all. That is NOT a failure of
        # your code: no test in the suite matches what was asked for. Some days have no
        # automated tests yet; judge those exercises by the checks written in the text.
        verdict = "no tests matched -- nothing to run here, this is not a failure"
    else:
        verdict = "not yet -- keep going"
    print()
    print(verdict)


def edit(relpath):
    """Print the absolute path of a source file, so you can open it in the editor."""
    print(OPTLAB / "src" / "optlab" / relpath)


check("tests/test_no_oracle_in_src.py")   # provided, and already green

---

## Exercise 0 — numpy warm-up  *(≈ 25 min)*

numpy was barely used in Week 1, and everything after this assumes fluency. No file to
edit: work in the cell below until all the assertions pass.

The one that matters most is the last: **compute `Xᵀ(Xw − y)` without a Python loop.**
That expression is the gradient of least squares, and you will write it for real in
Exercise 3.

In [ ]:
import numpy as np

rng = np.random.default_rng(0)
X = rng.normal(size=(6, 3))
w = rng.normal(size=3)
y = rng.normal(size=6)

# 1. Shapes. Predict each one BEFORE running.
assert (X @ w).shape == (6,)
assert (X.T @ X).shape == (3, 3)

# 2. Broadcasting: scale every COLUMN of X by the entries of d.
d = np.array([1.0, 10.0, 100.0])
assert np.allclose(X * d, X @ np.diag(d))

# 3. Broadcasting: scale every ROW of X by the entries of s.
s = rng.normal(size=6)
assert np.allclose(X * s[:, None], np.diag(s) @ X)

# 4. Reductions along an axis.
assert X.mean(axis=0).shape == (3,)     # one mean per column
assert X.mean(axis=1).shape == (6,)     # one mean per row

# 5. @ is matrix product, * is elementwise. They are not interchangeable.
assert not np.allclose(X.T @ X, (X.T * X.T).sum())

# 6. Solving beats inverting.
A = X.T @ X + np.eye(3)
b = X.T @ y
assert np.allclose(np.linalg.solve(A, b), np.linalg.inv(A) @ b)

# 7. THE ONE THAT MATTERS: no Python loop allowed.
loop = np.zeros(3)
for i in range(6):
    loop += X[i] * (X[i] @ w - y[i])
vectorized = ...                         # <-- your turn, one expression
assert np.allclose(vectorized, loop), "not equal yet"
print("all good")

---

## Exercise 1 — Finite differences  *(≈ 35 min)*

Write `numerical_gradient`, `numerical_jacobian` and `check_gradient`.

Use **central** differences: the error is `O(h²)` instead of `O(h)`, for one extra
evaluation per coordinate. `check_gradient` must return the relative error **and raise
`AssertionError`** when it exceeds `tol` — a check that cannot fail is not a check.

Then reproduce the U-curve from the lecture: sweep `h` from `1e-1` down to `1e-14` on a
function whose derivative you know, and find where rounding starts to beat truncation.

**Open:** `src/optlab/numerics/gradcheck.py`

In [ ]:
edit("numerics/gradcheck.py")
check("tests/unit/test_day1_gradcheck.py")

<details>
<summary><b>Plan B</b> — open only if you are stuck for more than ten minutes</summary>

Take one coordinate at a time: build a zero vector, put `h` in slot `i`, and evaluate `f` at `x + step` and `x - step`. `numerical_jacobian` is the same loop, except `f` returns a vector, so you stack the columns.

</details>

---

## Exercise 2 — Pointwise losses  *(≈ 25 min)*

Implement `SquaredError` and `LogisticNLL` in `losses.py`: `value`, `d1`, `d2`,
all vectorized over arrays.

`SquaredError` is `½(z − y)²`. `LogisticNLL` is `log(1 + eᶻ) − yz`, and it is the one
with the trap — computed naively, `z = 1000` gives `inf` when the true value is `1000`.
Use the stable softplus from the lecture.

Leave `Huber` and `PoissonNLL` alone; they are day 6.

**Open:** `src/optlab/losses.py`

In [ ]:
edit("losses.py")
check("tests/contracts/test_pointwise_loss_contract.py", "-m", "day1")

<details>
<summary><b>Plan B</b> — open only if you are stuck for more than ten minutes</summary>

`softplus(z) = max(z, 0) + log1p(exp(-abs(z)))`. For the derivatives: `d1 = sigmoid(z) - y` and `d2 = sigmoid(z) * (1 - sigmoid(z))` — predicted minus observed, and the variance of a Bernoulli.

</details>

---

## Exercise 3 — One GLM class for every likelihood  *(≈ 30 min)*

Implement `GLMLoss.value` and `GLMLoss.gradient`, plus the `linear_regression` and
`logistic_regression` helpers.

`value` is the mean of `φ(Xw, y)`. `gradient` is `Xᵀ φ'(Xw, y) / n` — the expression you
vectorized in Exercise 0.

Write it **once**: the same two lines serve linear, logistic, Poisson and robust
regression, because the only thing that changes is the injected `IPointwiseLoss`. Resist
any urge to branch on which loss it is.

Leave `hessian`, `n_samples` and `batch_gradient` — days 4 and 3.

**Open:** `src/optlab/problems/glm.py`

In [ ]:
edit("problems/glm.py")
check("tests/unit/test_day1_losses.py")

<details>
<summary><b>Plan B</b> — open only if you are stuck for more than ten minutes</summary>

`z = self.X @ w`, then `float(np.mean(self.pointwise.value(z, self.y)))` and `self.X.T @ self.pointwise.d1(z, self.y) / len(self.y)`.

</details>

---

## Exercise 4 — Two test problems  *(≈ 20 min)*

Implement `Quadratic` (`value`, `gradient`, and the `ill_conditioned` factory) and
`Rosenbrock` (`value`, `gradient`). Leave both `hessian` methods for day 4.

`Quadratic.ill_conditioned(n, kappa)` should have eigenvalues spaced logarithmically
between `1` and `kappa`, and `b = 0` so the minimizer is the origin and the error is just
`‖x‖`. You will lean on that convenience every day this week.

Now run the **contract test**: one test, every `IObjective`, checking that `gradient` really
is the derivative of `value`.

**Open:** `src/optlab/problems/quadratic.py`, `src/optlab/problems/rosenbrock.py`

In [ ]:
check("tests/contracts/test_objective_contract.py")

<details>
<summary><b>Plan B</b> — open only if you are stuck for more than ten minutes</summary>

Quadratic: `0.5 * x @ self.A @ x - self.b @ x` and `self.A @ x - self.b`. For the factory, `np.diag(np.logspace(0, np.log10(kappa), n))`. Rosenbrock's gradient is fiddly — write it, then let `check_gradient` tell you whether you got it right.

</details>

---

## Exercise 5 — Conditioning is a property of your data  *(≈ 20 min)*

Implement `standardize` in `datasets/load.py`, then measure what it does.

Build `A = XᵀX / n` on raw data with features on wildly different scales, compute
`np.linalg.cond(A)` before and after standardizing, and look at the ratio.

Nothing about the model changed. Only the units did. Write down the two condition numbers
— tomorrow you will predict the number of gradient-descent iterations from them, and the
prediction will be right.

**Open:** `optlab/datasets/load.py`

In [ ]:
import numpy as np
n = 500
f1 = rng.normal(size=n)
f2 = 0.3 * f1 + rng.normal(size=n)
X_raw = np.column_stack([f1, 1e-3 * f2])

kappa = lambda M: np.linalg.cond(M.T @ M / len(M))
std = lambda M: (M - M.mean(axis=0)) / M.std(axis=0)

print(f"raw          : {kappa(X_raw):12.1f}")
print(f"standardized : {kappa(std(X_raw)):12.1f}")
print(f"ratio        : {kappa(X_raw) / kappa(std(X_raw)):12.0f}x harder for free")

---

## Checkpoint

Everything from day 1 to day 1 should be green before you leave, and
`mypy --strict` must be clean. A red type check counts as a failure.

In [ ]:
check("-m", "day1")

In [ ]:
r = subprocess.run([sys.executable, "-m", "mypy"], cwd=OPTLAB,
                   capture_output=True, text=True)
print(r.stdout.strip() or r.stderr.strip())

---

## Before the debrief

Be able to answer:

1. Why is squared error the *Gaussian* maximum-likelihood estimator — and what would you
   use instead if you did not believe the noise was Gaussian?
2. Why does rescaling a feature change `κ`, when it cannot change what the model predicts?
3. Where is the bottom of your finite-difference U-curve, and why is it not at `h = 0`?

If your Week 1 autodiff module is in `src/optlab/autodiff/`, check one gradient against
`autodiff_gradient` as well. Two independent oracles catch different mistakes.